In [ ]:
import torch
import numpy as np
import uproot
import awkward as ak
import matplotlib.pyplot as plt
import sys
sys.path.append('../utils/')
sys.path.append('..')
import plotting_utils as pu
import data_utils as du
from of_dataset import OfDataset
from lightning_module import LOfTransformer

In [ ]:
# Get data and pseudodata files
mc_file = uproot.open('/global/cfs/cdirs/m3246/ZjetOmnifold/data/slimmed_files/WithTracks_ZjetOmnifold_May19_MGPy8FxFxRew_syst_train_Mar1023.root')
pd_file = uproot.open('/global/cfs/cdirs/m3246/ZjetOmnifold/data/slimmed_files/WithTracks_ZjetOmnifold_Aug5_PseudoDataSRew_Apr8_1_All.root')
mc_tree = mc_file['OmniTree']
pd_tree = pd_file['OmniTree']

In [ ]:
# Get pass190 for MC
mc_pass190 = ak.to_numpy(mc_tree['pass190'].array())

In [ ]:
# Get kinematics and one hots for MC
mc_kinematics, mc_indeces = du.get_kinematics(mc_tree, filter=mc_pass190)

In [ ]:
# Produe dummy kinematics and indeces for the pseudodata
pd_pass190 = ak.to_numpy(pd_tree['pass190'].array())
num_pd = len(pd_pass190)
pd_kinematics = ak.from_numpy(np.zeros(shape=(num_pd, 4, 4), dtype=np.float32))
pd_indeces = ak.from_numpy(np.zeros(shape=(num_pd, 4, 7), dtype=np.int32))

In [ ]:
# Make weights
mc_weights = np.ones(len(mc_kinematics))
pd_weights = np.ones(num_pd)

In [ ]:
# Make labels
mc_labels = np.zeros((len(mc_kinematics), 1), dtype=np.int32)
pd_labels = np.ones((num_pd, 1), dtype=np.int32)

In [ ]:
# Make dummy plotting
mc_plotting = np.zeros((len(mc_kinematics), 1), dtype=np.int32)
pd_plotting = np.ones((num_pd, 1), dtype=np.int32)

In [ ]:
# Concatenate all of the data
kinematics = ak.concatenate((mc_kinematics, pd_kinematics), axis=0)
indeces = ak.concatenate((mc_indeces, pd_indeces), axis=0)
weights = np.concatenate((mc_weights, pd_weights), axis=0)
labels = np.concatenate((mc_labels, pd_labels), axis=0)
plotting = np.concatenate((mc_plotting, pd_plotting), axis=0)

In [ ]:
print(len(kinematics))
print(len(indeces))
print(len(weights))
print(len(labels))
print(len(plotting))

In [ ]:
# Make pytorch datasets
all_dataset = OfDataset(kinematics, labels, weights, plotting, object_indeces=indeces, max_tracks=264)

In [ ]:
# Do train / val split
generator = torch.Generator().manual_seed(690)
train_dataset, val_dataset = torch.utils.data.random_split(all_dataset, [0.8, 0.2], generator=generator)

In [ ]:
# Print the muon pT for the first event in both sets
train_muon_pt = train_dataset[0][0][0,0,:2]
val_muon_pt = val_dataset[0][0][0,0,:2]
print("Train muon pt: {} {}".format(train_muon_pt[0], train_muon_pt[1]))
print("Val muon pt: {} {}".format(val_muon_pt[0], val_muon_pt[1]))

In [ ]:
# Load trained model
model = LOfTransformer.load_from_checkpoint('../lightning_logs/version_27469429/checkpoints/epoch=4-val_loss=0.6110.ckpt')
model.cpu()
model.eval()

In [ ]:
# Run inference over the first event in the training / validation sets
train_label = train_dataset[0][1]
train_output = model(train_dataset[0][0], train_dataset[0][2])
val_label = val_dataset[0][1]
val_output = model(val_dataset[0][0], val_dataset[0][2])

In [ ]:
print(train_output)
print(val_output)

In [ ]:
print(train_dataset[0][0][0,0,:2])
print(val_dataset[0][0][0,0,:2])